In [27]:

import torch
import numpy as np
from numpy.dtypes import StringDType
import csv
from random import randrange
import subprocess
import mmap

In [28]:
def countLines (file_path) -> int:
  line_count = 0
  with open(file_path, "r") as python_filehandle:
    with mmap.mmap(python_filehandle.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_filehandle:
        while mmap_filehandle.readline():
            line_count += 1

  return line_count

In [29]:
def retrieveSentence (sentence_idx, tokens_list, sentence_offsets) -> str:
    return "".join(tokens_dict[tokno].replace("<wb>", " ") for tokno in tokens_list[sentence_offsets[sentence_idx]:sentence_offsets[sentence_idx+1]]).strip()

In [30]:
token_vocab_length = countLines("bpe_token_indices.csv")

In [31]:
bpe_token_indices_file = open("bpe_token_indices.csv", "r")
tokenised_chu_words_training_file = open("tokenised_chu_words_training_deepcleaned.csv", "r")

In [32]:
tokens_list = []
word_offsets = []
with mmap.mmap(tokenised_chu_words_training_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_tokens_file:
  word_token_count = 0
  for line in iter(mmap_tokens_file.readline, b""):
    word_offsets.append(word_token_count)
    for token_no in line.decode("utf-8").strip().split(",")[0].split(" "):
      tokens_list.append(int(token_no))
      word_token_count += 1

In [33]:
tokens_tensor = torch.tensor(tokens_list, dtype=torch.float32)

In [34]:
sentence_offsets = []
row_no = 0
sentence_no_prev = 0
token_count = 0
for row in csv.DictReader(open("../../chu_words_tagged.csv", "r"), delimiter="|"):
    sentence_no = int(row["sentence_no"])
    if sentence_no != sentence_no_prev:
        sentence_offsets.append(word_offsets[row_no])
        sentence_no_prev = sentence_no
    row_no += 1

In [35]:
len(tokens_list), len(word_offsets), len(sentence_offsets)

(347761, 242536, 27241)

In [36]:
tokens_dict = {}
tokens_dict_reversed = {}
with mmap.mmap(bpe_token_indices_file.fileno(), length=0, access=mmap.ACCESS_READ) as mmap_bpe_file:
    for bin_line in iter(mmap_bpe_file.readline, b""):
        line = bin_line.decode("utf-8").strip()
        split_line = line.split(",")
        tokens_dict[int(split_line[0])] = split_line[1].strip()
        tokens_dict_reversed[split_line[1]] = int(split_line[0])

In [37]:
retrieveSentence(15995, tokens_list, sentence_offsets)

'ѣкоже бо богатъ нѣкто піръ велікъ гостемъ своімъ оуготовівъ і посълавъ по зъванъіѧ імъ ізъходѧште зъваніі къждо і своего домоу вѣдѧтъ ізвѣстьнѣ ѣко ідѫтъ на веселіе і съ оусрьдіемъ на піръ пріходѧтъ въшедъше на обѣдъ і сѣдъше на трепезѣ въкоусівъше віна да аште добро бѫдетъ то сладьцѣ піѭтъ отъ него і вьзвеселѧтъ сѧ і отъ оупітіѣ не могѫтъ іті въ домъі своѧ понеже много сѫтъ пілі въсладівъше сі віномъ доньдеже прідѫтъ своі імъ і імше пріведѫтъ въ домъі своѧ іже заоутра въстаѭште егда ісѫчѧтъ віно радоуѭтъ сѧ зѣло ѣко оу своіхъ сѫтъ сі дома лежалі такожде і рабі хрістосові егда відѧтъ ѣко зовѫтъ ѧ сѫдіѧ на сѫдіште то вѣдѧтъ ѣко на троудъ і на ноуждѫ ідѫтъ'

In [38]:
tokens_list[0:10], word_offsets[0:10], sentence_offsets[0:10]

([47, 2264, 1293, 675, 1849, 85, 95, 19, 218, 780],
 [0, 1, 4, 9, 11, 12, 13, 15, 16, 18],
 [0, 11, 36, 62, 110, 116, 179, 188, 208, 218])

In [43]:
sentence_offsets_tensor = torch.tensor(sentence_offsets, dtype=torch.int64)
tensor_snt_lngths = torch.diff(sentence_offsets_tensor)
print("Max sentence length:", tensor_snt_lngths.max())
print("Median sentence length:", tensor_snt_lngths.median())
tensor_snt_lngths[3299] = 0
tensor_snt_lngths[14044] = 0
tensor_snt_lngths[21318] = 0
print(tensor_snt_lngths.max())

Max sentence length: tensor(291)
Median sentence length: tensor(10)
tensor(167)


In [44]:
tensor_snt_lngths = torch.diff(sentence_offsets_tensor)
for i in range(len(tensor_snt_lngths)):
    if tensor_snt_lngths[i] == 167:
        print(i)

15995


In [52]:
token_embedder = torch.nn.Embedding(num_embeddings=4539, embedding_dim=256, padding_idx=0)

In [53]:
token_embeddings = token_embedder(torch.tensor(list(tokens_dict.keys())))

In [59]:
token_embeddings[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 